# Customer Retention & Churn Analytics — SQL Analysis

## SQL Business Analysis

This notebook uses SQLite to analyze the cleaned and ML-scored customer dataset from a business perspective. The analysis covers overall churn KPIs, contracts, payment methods, services, tenure, predicted risk, retention priority, financial exposure, and customer-level retention candidates.

In [1]:
import sqlite3
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Connect to the existing SQLite database
conn = sqlite3.connect("customer_churn.db")

print("SQLite connection established successfully.")


SQLite connection established successfully.


In [2]:
tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    conn
)

tables


,name
0,customer_churn


In [3]:
query = """
SELECT
    COUNT(*) AS Total_Rows,
    COUNT(DISTINCT customerID) AS Unique_Customers
FROM customer_churn;
"""

pd.read_sql_query(query, conn)

,Total_Rows,Unique_Customers
0,7043,7043


In [4]:
query = """
PRAGMA table_info(customer_churn);
"""

table_structure = pd.read_sql_query(query, conn)

table_structure[["name", "type"]]

,name,type
0,customerID,TEXT
1,gender,TEXT
2,SeniorCitizen,INTEGER
3,Partner,TEXT
4,Dependents,TEXT
5,tenure,INTEGER
6,PhoneService,TEXT
7,MultipleLines,TEXT
8,InternetService,TEXT
9,OnlineSecurity,TEXT


## 7.2 — Overall Customer & Churn KPIs

This section establishes the overall customer base, churn volume, churn rate, and monthly-charge exposure associated with churned customers.

In [5]:
query = """
SELECT
    COUNT(*) AS Total_Customers,

    SUM(CASE
        WHEN Churn = 'Yes' THEN 1
        ELSE 0
    END) AS Churned_Customers,

    SUM(CASE
        WHEN Churn = 'No' THEN 1
        ELSE 0
    END) AS Retained_Customers,

    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS Churn_Rate_Pct,

    ROUND(SUM(MonthlyCharges), 2) AS Total_Monthly_Charges,

    ROUND(
        SUM(CASE
            WHEN Churn = 'Yes' THEN MonthlyCharges
            ELSE 0
        END),
        2
    ) AS Churn_Associated_Monthly_Charges

FROM customer_churn;
"""

overall_kpis = pd.read_sql_query(query, conn)
overall_kpis

,Total_Customers,Churned_Customers,Retained_Customers,Churn_Rate_Pct,Total_Monthly_Charges,Churn_Associated_Monthly_Charges
0,7043,1869,5174,26.54,"456,116.60","139,130.85"


## 7.3 — Contract Analysis

Contract groups are compared using customer volume, churn rate, and monthly-charge exposure to identify contract structures associated with elevated observed churn.

In [6]:
query = """
SELECT
    Contract,

    COUNT(*) AS Customers,

    SUM(CASE
        WHEN Churn = 'Yes' THEN 1
        ELSE 0
    END) AS Churned_Customers,

    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS Churn_Rate_Pct,

    ROUND(SUM(MonthlyCharges), 2) AS Total_Monthly_Charges,

    ROUND(
        SUM(CASE
            WHEN Churn = 'Yes' THEN MonthlyCharges
            ELSE 0
        END),
        2
    ) AS Churn_Associated_Monthly_Charges

FROM customer_churn

GROUP BY Contract

ORDER BY Churn_Rate_Pct DESC;
"""

contract_analysis = pd.read_sql_query(query, conn)
contract_analysis

,Contract,Customers,Churned_Customers,Churn_Rate_Pct,Total_Monthly_Charges,Churn_Associated_Monthly_Charges
0,Month-to-month,3875,1655,42.71,"257,294.15","120,847.10"
1,One year,1473,166,11.27,"95,816.60","14,118.45"
2,Two year,1695,48,2.83,"103,005.85","4,165.30"


## 7.4 — Payment Method Analysis

Payment methods are evaluated using customer count, observed churn rate, and monthly-charge exposure to identify payment segments associated with higher retention risk.

In [7]:
query = """
SELECT
    PaymentMethod,

    COUNT(*) AS Customers,

    SUM(CASE
        WHEN Churn = 'Yes' THEN 1
        ELSE 0
    END) AS Churned_Customers,

    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS Churn_Rate_Pct,

    ROUND(SUM(MonthlyCharges), 2) AS Total_Monthly_Charges,

    ROUND(
        SUM(CASE
            WHEN Churn = 'Yes' THEN MonthlyCharges
            ELSE 0
        END),
        2
    ) AS Churn_Associated_Monthly_Charges

FROM customer_churn

GROUP BY PaymentMethod

ORDER BY Churn_Rate_Pct DESC;
"""

payment_analysis = pd.read_sql_query(query, conn)
payment_analysis

,PaymentMethod,Customers,Churned_Customers,Churn_Rate_Pct,Total_Monthly_Charges,Churn_Associated_Monthly_Charges
0,Electronic check,2365,1071,45.29,"180,345.00","84,288.75"
1,Mailed check,1612,308,19.11,"70,794.30","16,803.60"
2,Bank transfer (automatic),1544,258,16.71,"103,745.45","20,091.90"
3,Credit card (automatic),1522,232,15.24,"101,231.85","17,946.60"


## 7.5 — Customer Services Analysis

Internet service, Online Security, and Tech Support are analyzed to identify service-related customer segments associated with elevated observed churn.

In [8]:
query = """
SELECT
    InternetService,

    COUNT(*) AS Customers,

    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        AS Churned_Customers,

    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS Churn_Rate_Pct

FROM customer_churn

GROUP BY InternetService

ORDER BY Churn_Rate_Pct DESC;
"""

internet_service_analysis = pd.read_sql_query(query, conn)
internet_service_analysis

,InternetService,Customers,Churned_Customers,Churn_Rate_Pct
0,Fiber optic,3096,1297,41.89
1,DSL,2421,459,18.96
2,No,1526,113,7.40


In [9]:
query = """
SELECT
    InternetService,

    COUNT(*) AS Customers,

    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        AS Churned_Customers,

    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS Churn_Rate_Pct

FROM customer_churn

GROUP BY InternetService

ORDER BY Churn_Rate_Pct DESC;
"""

internet_service_analysis = pd.read_sql_query(query, conn)
internet_service_analysis

,InternetService,Customers,Churned_Customers,Churn_Rate_Pct
0,Fiber optic,3096,1297,41.89
1,DSL,2421,459,18.96
2,No,1526,113,7.40


In [10]:
query = """
SELECT
    TechSupport,

    COUNT(*) AS Customers,

    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        AS Churned_Customers,

    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS Churn_Rate_Pct

FROM customer_churn

GROUP BY TechSupport

ORDER BY Churn_Rate_Pct DESC;
"""

tech_support_analysis = pd.read_sql_query(query, conn)
tech_support_analysis

,TechSupport,Customers,Churned_Customers,Churn_Rate_Pct
0,No,3473,1446,41.64
1,Yes,2044,310,15.17
2,No internet service,1526,113,7.40


## 7.6 — Customer Tenure Analysis

Customers are grouped by tenure to examine how observed churn varies across the customer lifecycle.

In [11]:
query = """
WITH tenure_segments AS (

    SELECT
        CASE
            WHEN tenure <= 12 THEN '0-12 months'
            WHEN tenure <= 24 THEN '13-24 months'
            WHEN tenure <= 36 THEN '25-36 months'
            WHEN tenure <= 48 THEN '37-48 months'
            WHEN tenure <= 60 THEN '49-60 months'
            ELSE '61-72 months'
        END AS Tenure_Group,

        CASE
            WHEN tenure <= 12 THEN 1
            WHEN tenure <= 24 THEN 2
            WHEN tenure <= 36 THEN 3
            WHEN tenure <= 48 THEN 4
            WHEN tenure <= 60 THEN 5
            ELSE 6
        END AS Tenure_Order,

        Churn,
        MonthlyCharges

    FROM customer_churn
)

SELECT
    Tenure_Group,

    COUNT(*) AS Customers,

    SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        AS Churned_Customers,

    ROUND(
        100.0 * SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS Churn_Rate_Pct,

    ROUND(SUM(MonthlyCharges), 2)
        AS Total_Monthly_Charges

FROM tenure_segments

GROUP BY Tenure_Group, Tenure_Order

ORDER BY Tenure_Order;
"""

tenure_analysis = pd.read_sql_query(query, conn)
tenure_analysis

,Tenure_Group,Customers,Churned_Customers,Churn_Rate_Pct,Total_Monthly_Charges
0,0-12 months,2186,1037,47.44,"122,629.75"
1,13-24 months,1024,294,28.71,"62,829.85"
2,25-36 months,832,180,21.63,"54,558.80"
3,37-48 months,762,145,19.03,"50,534.50"
4,49-60 months,832,120,14.42,"58,698.25"
5,61-72 months,1407,93,6.61,"106,865.45"


## 7.7 — ML Risk Category Analysis

Full-population model scores are summarized using project-defined churn-risk categories to examine customer volume, average predicted risk, and monthly-charge exposure.

In [12]:
query = """
SELECT
    Risk_Category,

    COUNT(*) AS Customers,

    ROUND(
        100.0 * COUNT(*) /
        (SELECT COUNT(*) FROM customer_churn),
        2
    ) AS Customer_Share_Pct,

    ROUND(AVG(Churn_Probability), 4)
        AS Avg_Churn_Probability,

    ROUND(SUM(MonthlyCharges), 2)
        AS Monthly_Charges,

    ROUND(AVG(MonthlyCharges), 2)
        AS Avg_Monthly_Charge

FROM customer_churn

GROUP BY Risk_Category

ORDER BY
    CASE Risk_Category
        WHEN 'Very High Risk' THEN 1
        WHEN 'High Risk' THEN 2
        WHEN 'Medium Risk' THEN 3
        WHEN 'Low Risk' THEN 4
    END;
"""

risk_analysis = pd.read_sql_query(query, conn)
risk_analysis

,Risk_Category,Customers,Customer_Share_Pct,Avg_Churn_Probability,Monthly_Charges,Avg_Monthly_Charge
0,Very High Risk,292,4.15,0.85,"24,053.45",82.37
1,High Risk,706,10.02,0.69,"56,108.70",79.47
2,Medium Risk,1604,22.77,0.44,"116,208.65",72.45
3,Low Risk,4441,63.06,0.10,"259,745.80",58.49


## 7.8 — Retention Priority Analysis

Customers are summarized by the project-defined Priority Group, which combines churn probability with normalized MonthlyCharges to support business-oriented retention prioritization.

In [13]:
query = """
SELECT
    Priority_Group,

    COUNT(*) AS Customers,

    ROUND(
        100.0 * COUNT(*) /
        (SELECT COUNT(*) FROM customer_churn),
        2
    ) AS Customer_Share_Pct,

    ROUND(AVG(Priority_Score), 4)
        AS Avg_Priority_Score,

    ROUND(AVG(Churn_Probability), 4)
        AS Avg_Churn_Probability,

    ROUND(SUM(MonthlyCharges), 2)
        AS Monthly_Charges

FROM customer_churn

GROUP BY Priority_Group

ORDER BY
    CASE Priority_Group
        WHEN 'Priority 1' THEN 1
        WHEN 'Priority 2' THEN 2
        WHEN 'Priority 3' THEN 3
    END;
"""

priority_analysis = pd.read_sql_query(query, conn)
priority_analysis

,Priority_Group,Customers,Customer_Share_Pct,Avg_Priority_Score,Avg_Churn_Probability,Monthly_Charges
0,Priority 1,612,8.69,0.77,0.79,"53,111.25"
1,Priority 2,1148,16.30,0.59,0.55,"94,279.50"
2,Priority 3,5283,75.01,0.25,0.14,"308,725.85"


## 7.9 — Financial Exposure Analysis

Model-estimated churn probabilities are combined with MonthlyCharges to quantify probability-weighted financial exposure. These values represent analytical exposure indicators rather than guaranteed future revenue loss.

In [14]:
query = """
SELECT
    ROUND(SUM(MonthlyCharges), 2)
        AS Total_Monthly_Charges,

    ROUND(
        SUM(Churn_Probability * MonthlyCharges),
        2
    ) AS Probability_Weighted_Exposure,

    ROUND(
        100.0 *
        SUM(Churn_Probability * MonthlyCharges)
        / SUM(MonthlyCharges),
        2
    ) AS Weighted_Exposure_Pct

FROM customer_churn;
"""

financial_exposure = pd.read_sql_query(query, conn)
financial_exposure

,Total_Monthly_Charges,Probability_Weighted_Exposure,Weighted_Exposure_Pct
0,"456,116.60","139,189.02",30.52


In [15]:
query = """
SELECT
    Risk_Category,

    COUNT(*) AS Customers,

    ROUND(SUM(MonthlyCharges), 2)
        AS Monthly_Charges,

    ROUND(
        SUM(Churn_Probability * MonthlyCharges),
        2
    ) AS Probability_Weighted_Exposure

FROM customer_churn

GROUP BY Risk_Category

ORDER BY
    CASE Risk_Category
        WHEN 'Very High Risk' THEN 1
        WHEN 'High Risk' THEN 2
        WHEN 'Medium Risk' THEN 3
        WHEN 'Low Risk' THEN 4
    END;
"""

risk_exposure = pd.read_sql_query(query, conn)
risk_exposure

,Risk_Category,Customers,Monthly_Charges,Probability_Weighted_Exposure
0,Very High Risk,292,"24,053.45","20,477.66"
1,High Risk,706,"56,108.70","38,831.30"
2,Medium Risk,1604,"116,208.65","51,339.15"
3,Low Risk,4441,"259,745.80","28,540.91"


## 7.10 — Top Retention Candidates

Priority 1 customers are ranked using Priority Score, Churn Probability, and MonthlyCharges to identify a manageable retention-candidate queue. The Top 50 threshold is a project-level operational choice rather than a universal business rule.

In [16]:
query = """
WITH ranked_customers AS (

    SELECT
        customerID,
        tenure,
        Contract,
        PaymentMethod,
        InternetService,
        OnlineSecurity,
        TechSupport,
        MonthlyCharges,
        Churn_Probability,
        Risk_Category,
        Priority_Score,
        Priority_Group,

        ROW_NUMBER() OVER (
            ORDER BY
                Priority_Score DESC,
                Churn_Probability DESC,
                MonthlyCharges DESC
        ) AS Retention_Rank

    FROM customer_churn

    WHERE Priority_Group = 'Priority 1'
)

SELECT *
FROM ranked_customers

WHERE Retention_Rank <= 50

ORDER BY Retention_Rank;
"""

top_retention_candidates = pd.read_sql_query(query, conn)
top_retention_candidates

,customerID,tenure,Contract,PaymentMethod,InternetService,OnlineSecurity,TechSupport,MonthlyCharges,Churn_Probability,Risk_Category,Priority_Score,Priority_Group,Retention_Rank
0,7216-EWTRS,1,Month-to-month,Electronic check,Fiber optic,No,No,100.80,0.94,Very High Risk,0.91,Priority 1,1
1,5419-JPRRN,1,Month-to-month,Electronic check,Fiber optic,No,No,101.45,0.93,Very High Risk,0.91,Priority 1,2
2,0107-YHINA,1,Month-to-month,Electronic check,Fiber optic,No,No,99.75,0.93,Very High Risk,0.90,Priority 1,3
3,3178-FESZO,1,Month-to-month,Credit card (automatic),Fiber optic,No,No,100.25,0.91,Very High Risk,0.89,Priority 1,4
4,4910-GMJOT,1,Month-to-month,Electronic check,Fiber optic,No,No,94.60,0.93,Very High Risk,0.89,Priority 1,5
5,9300-AGZNL,1,Month-to-month,Electronic check,Fiber optic,No,No,94.00,0.93,Very High Risk,0.89,Priority 1,6
6,8149-RSOUN,1,Month-to-month,Electronic check,Fiber optic,No,No,93.85,0.93,Very High Risk,0.89,Priority 1,7
7,9497-QCMMS,1,Month-to-month,Electronic check,Fiber optic,No,No,93.55,0.93,Very High Risk,0.89,Priority 1,8
8,5178-LMXOP,1,Month-to-month,Electronic check,Fiber optic,No,No,95.10,0.92,Very High Risk,0.88,Priority 1,9
9,0295-PPHDO,1,Month-to-month,Electronic check,Fiber optic,No,No,95.45,0.91,Very High Risk,0.88,Priority 1,10


In [17]:
query = """
SELECT
    COUNT(*) AS Critical_Customers,

    ROUND(AVG(Churn_Probability), 4)
        AS Avg_Churn_Probability,

    ROUND(AVG(Priority_Score), 4)
        AS Avg_Priority_Score,

    ROUND(SUM(MonthlyCharges), 2)
        AS Monthly_Charges,

    ROUND(
        SUM(Churn_Probability * MonthlyCharges),
        2
    ) AS Probability_Weighted_Exposure

FROM customer_churn

WHERE Priority_Group = 'Priority 1'

AND Risk_Category IN (
    'High Risk',
    'Very High Risk'
);
"""

critical_retention_pool = pd.read_sql_query(query, conn)
critical_retention_pool

,Critical_Customers,Avg_Churn_Probability,Avg_Priority_Score,Monthly_Charges,Probability_Weighted_Exposure
0,612,0.79,0.77,"53,111.25","41,725.39"


In [18]:
query = """
WITH customer_risk_factors AS (

    SELECT
        customerID,
        Churn_Probability,
        Risk_Category,
        Priority_Score,
        Priority_Group,
        MonthlyCharges,

        (
            CASE WHEN Contract = 'Month-to-month' THEN 1 ELSE 0 END +
            CASE WHEN PaymentMethod = 'Electronic check' THEN 1 ELSE 0 END +
            CASE WHEN tenure <= 12 THEN 1 ELSE 0 END +
            CASE WHEN OnlineSecurity = 'No' THEN 1 ELSE 0 END +
            CASE WHEN TechSupport = 'No' THEN 1 ELSE 0 END
        ) AS Descriptive_Risk_Factor_Count

    FROM customer_churn
)

SELECT *

FROM customer_risk_factors

WHERE Priority_Group = 'Priority 1'

ORDER BY
    Priority_Score DESC,
    Churn_Probability DESC,
    MonthlyCharges DESC

LIMIT 50;
"""

retention_risk_factors = pd.read_sql_query(query, conn)
retention_risk_factors

,customerID,Churn_Probability,Risk_Category,Priority_Score,Priority_Group,MonthlyCharges,Descriptive_Risk_Factor_Count
0,7216-EWTRS,0.94,Very High Risk,0.91,Priority 1,100.80,5
1,5419-JPRRN,0.93,Very High Risk,0.91,Priority 1,101.45,5
2,0107-YHINA,0.93,Very High Risk,0.90,Priority 1,99.75,5
3,3178-FESZO,0.91,Very High Risk,0.89,Priority 1,100.25,4
4,4910-GMJOT,0.93,Very High Risk,0.89,Priority 1,94.60,5
5,9300-AGZNL,0.93,Very High Risk,0.89,Priority 1,94.00,5
6,8149-RSOUN,0.93,Very High Risk,0.89,Priority 1,93.85,5
7,9497-QCMMS,0.93,Very High Risk,0.89,Priority 1,93.55,5
8,5178-LMXOP,0.92,Very High Risk,0.88,Priority 1,95.10,5
9,0295-PPHDO,0.91,Very High Risk,0.88,Priority 1,95.45,5


## 7.11 — Final SQL Business Summary

The final SQL layer consolidates historical churn, predictive risk, customer priority, and financial exposure into an executive-level business summary. Historical churn measures and model-generated risk measures are intentionally kept conceptually separate.

In [19]:
query = """
SELECT
    COUNT(*) AS Total_Customers,

    SUM(CASE
        WHEN Churn = 'Yes' THEN 1
        ELSE 0
    END) AS Historical_Churned_Customers,

    ROUND(
        100.0 *
        SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END)
        / COUNT(*),
        2
    ) AS Historical_Churn_Rate_Pct,

    SUM(CASE
        WHEN Risk_Category IN ('High Risk', 'Very High Risk')
        THEN 1 ELSE 0
    END) AS Higher_Risk_Customers,

    SUM(CASE
        WHEN Priority_Group = 'Priority 1'
        THEN 1 ELSE 0
    END) AS Priority_1_Customers,

    ROUND(SUM(MonthlyCharges), 2)
        AS Total_Monthly_Charges,

    ROUND(
        SUM(CASE
            WHEN Churn = 'Yes'
            THEN MonthlyCharges
            ELSE 0
        END),
        2
    ) AS Historical_Churn_Associated_Charges,

    ROUND(
        SUM(CASE
            WHEN Risk_Category IN ('High Risk', 'Very High Risk')
            THEN MonthlyCharges
            ELSE 0
        END),
        2
    ) AS Higher_Risk_Monthly_Charges,

    ROUND(
        SUM(Churn_Probability * MonthlyCharges),
        2
    ) AS Probability_Weighted_Exposure,

    ROUND(
        SUM(CASE
            WHEN Priority_Group = 'Priority 1'
            THEN Churn_Probability * MonthlyCharges
            ELSE 0
        END),
        2
    ) AS Priority_1_Weighted_Exposure

FROM customer_churn;
"""

executive_sql_summary = pd.read_sql_query(query, conn)

executive_sql_summary.T

,0
Total_Customers,"7,043.00"
Historical_Churned_Customers,"1,869.00"
Historical_Churn_Rate_Pct,26.54
Higher_Risk_Customers,998.00
Priority_1_Customers,612.00
Total_Monthly_Charges,"456,116.60"
Historical_Churn_Associated_Charges,"139,130.85"
Higher_Risk_Monthly_Charges,"80,162.15"
Probability_Weighted_Exposure,"139,189.02"
Priority_1_Weighted_Exposure,"41,725.39"


## SQL Analysis — Key Findings

SQL analysis confirmed that churn risk and financial exposure are unevenly distributed across the customer base. Month-to-month contracts, electronic-check payments, early-tenure customers, fiber-optic service, and customers without selected support services showed elevated observed churn.

ML-generated churn probabilities extended the historical analysis to customer-level risk assessment, while the project-defined Priority Score combined predicted risk with MonthlyCharges to support retention prioritization.

Priority 1 customers with High or Very High predicted risk form the most critical intervention population. Probability-weighted exposure provides financial context for prioritization but should not be interpreted as guaranteed revenue loss.

Overall, the SQL layer translates descriptive analysis and ML scoring into business-oriented customer segmentation, financial exposure analysis, and actionable retention-candidate ranking.

In [20]:
conn.close()

print("SQLite connection closed successfully.")

SQLite connection closed successfully.
